# ML Project

In [1]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from scipy.stats import spearmanr
import numpy as np
import itertools

import warnings
warnings.filterwarnings('ignore')

In [2]:
# After downloading the X_train/X_test/Y_train .csv files in your working directory:

X_df =pd.read_csv('X_train_NHkHMNU.csv', delimiter= ',')
y_df =pd.read_csv('y_train_ZAN5mwg.csv', delimiter= ',')
X_test_df =pd.read_csv('X_test_final.csv', delimiter= ',')

df = pd.merge(X_df,y_df,on='ID')

## Feature engineering
We want to add more columns to create more precise models so we will do some feature engineering 

We can compute rolling statistics to capture trends and fluctuations over time. The rolling mean smooths the data to highlight long-term trends, while the rolling standard deviation measures volatility and variability within specific time windows (weekly and monthly). The slope, calculated using linear regression on rolling windows, captures the rate of change or trend direction, helping to identify whether values are generally increasing or decreasing over time.

In [3]:
# Function to calculate rolling statistics (mean, std, median, min, max, slope) for DE_ and FR_ columns
def add_statistics(df, variables, windows):
    def slope(y):
        return np.polyfit(range(len(y)), y, 1)[0] if len(y) > 0 else np.nan

    for var in variables:
        for window in windows:
            for country in ['DE_', 'FR_']:
                col = f'{country}{var}'
                df[f'{col}_MEAN_{window}D'] = df[col].rolling(window=window).mean()
                df[f'{col}_STD_{window}D'] = df[col].rolling(window=window).std()
                df[f'{col}_MEDIAN_{window}D'] = df[col].rolling(window=window).median()
                df[f'{col}_MIN_{window}D'] = df[col].rolling(window=window).min()
                df[f'{col}_MAX_{window}D'] = df[col].rolling(window=window).max()
                df[f'{col}_SLOPE_{window}D'] = df[col].rolling(window=window).apply(slope, raw=True)
    return df


Then we can use technical indicators widely use in finance like volatility and moving averages that are used to analyze time-series data. Volatility, typically measured by rolling standard deviation, helps quantify the level of fluctuation or uncertainty in the data. Moving averages, especially exponential moving averages (EMA), are used to smooth short-term fluctuations and highlight long-term trends or cycles, making it easier to detect underlying patterns in the data.

In [4]:
def add_indicators(df):
    for commodity in ['GAS_RET', 'COAL_RET', 'CARBON_RET']:
        df[f'{commodity}_VOLATILITY_WEEKLY'] = df[commodity].rolling(window=7).std()
        df[f'{commodity}_VOLATILITY_MONTHLY'] = df[commodity].rolling(window=30).std()
        df[f'{commodity}_EMA_MONTHLY'] = df[commodity].ewm(span=30, adjust=False).mean()
    
    return df

Finally we can add column specific with this domain area such as energy ratios, weather effects, and clustering, wich will provide further insights into underlying patterns. Weather effects, such as temperature or wind, are crucial in energy demand prediction, as they directly influence consumption and production.

In [5]:
# Function to add energy source ratios and effects
def add_energy(df):
    energy_sources = ['GAS', 'COAL', 'HYDRO', 'NUCLEAR', 'SOLAR', 'WINDPOW']
    
    for country in ['DE_', 'FR_']:
        total_energy = sum(df[f'{country}{source}'] for source in energy_sources)
        
        for source in energy_sources:
            df[f'{country}{source}_RATIO'] = df[f'{country}{source}'] / total_energy

        df[f'{country}WIND_SOLAR'] = df[f'{country}WINDPOW'] + df[f'{country}SOLAR']
        df[f'{country}TEMP_EFFECT'] = df[f'{country}TEMP'] * df[f'{country}CONSUMPTION']
        df[f'{country}WIND_EFFECT'] = df[f'{country}WIND'] * df[f'{country}WINDPOW']
        df[f'{country}SOLAR_EFFECT'] = (df[f'{country}SOLAR'] / df[f'{country}TEMP']).replace([np.inf, -np.inf], np.nan).fillna(0)
    
    return df

Now that we have rearrange all our functions to add columns we can create a single function that call the others

In [6]:
# Main function to add custom features by combining all the previous functions
def add_columns(df):
    periods=[7, 30]# Weekly and Monthly
    variables = ['CONSUMPTION', 'GAS', 'COAL', 'HYDRO', 'NUCLEAR', 'SOLAR', 'WINDPOW', 'TEMP', 'RAIN', 'WIND']

    df=add_statistics(df, variables, periods)
    df=add_energy(df)
    df=add_indicators(df)
    
    df.fillna(df.mean(), inplace=True)
    return df

The following function calculates the correlation matrix and selects features that are either highly correlated with each other or have low correlation with the target variable. It combines both conditions to decide which features to drop

In [7]:
def select_features_based_on_correlation(df, target_column, multicollinear_threshold, correlation_threshold):
    # Calculate the Spearman correlation matrix because it is the metric that we choose
    corr_matrix=df.corr(method='spearman')
    # Identify columns that are highly correlated with each other
    # (excluding the target variable correlation)
    high_corr_var=np.where(corr_matrix > multicollinear_threshold)
    high_corr_var=[(corr_matrix.index[x], corr_matrix.columns[y]) 
                     for x, y in zip(*high_corr_var) 
                     if x!=y and x < y]
    # Extract the names of columns to drop based on multicollinearity
    multicollinear_features=set([item for sublist in high_corr_var for item in sublist])
    # Identify features that have a low correlation with the target variable
    low_corr_with_target=corr_matrix[target_column][abs(corr_matrix[target_column]) < correlation_threshold].index.tolist()
    # Combine features to drop due to multicollinearity and low correlation with target
    features_to_drop=multicollinear_features.union(low_corr_with_target)
    # Determine the final list of features to keep
    features_to_keep=[feature for feature in df.columns if feature not in features_to_drop and feature!=target_column]
    
    return features_to_keep

Here, we filter the dataset for France (COUNTRY == 'FR'), remove certain columns, and keep some of the original columns, which can still have an impact on our prediction even if they are less correlated than the ‘artificial’ columns from feature engineering.

In [8]:
df_fr = df[df['COUNTRY'] == 'FR']
df_fr = df_fr.drop(columns=['COUNTRY', 'FR_NET_EXPORT', 'ID'])
# We retain the columns in the main dataset, but not the columns with a high correlation with the others or a lower correlation with TARGET
col_fr_best = select_features_based_on_correlation(df_fr, 'TARGET', 0.93, 0.08)

# Select columns that do not start with 'DE' (to avoid mixing German data)
df_no_de = df_fr[df_fr.columns[~df_fr.columns.str.startswith('DE')]]
col_fr = [col for col in df_no_de.columns if col in col_fr_best]
df_fr.fillna(df_fr.mean(), inplace=True)

Same for Germany

In [9]:
df_de = df[df['COUNTRY'] == 'DE']
df_de = df_de.drop(columns=['COUNTRY', 'DE_NET_EXPORT', 'ID'])
# We retain the columns in the main dataset, but not the columns with a high correlation with the others or a lower correlation with TARGET
col_de_best = select_features_based_on_correlation(df_de, 'TARGET', 0.93, 0.08)

# Select columns that do not start with 'FR' (to avoid mixing French data)
df_no_fr = df_de[df_de.columns[~df_de.columns.str.startswith('FR')]]
col_de = [col for col in df_no_fr.columns if col in col_de_best]
df_de.fillna(df_de.mean(), inplace=True)

In [10]:
# Extract the original columns
original_fr = df_fr[col_fr]
original_de = df_de[col_de]

In [11]:
# Apply feature engineering
df_fr = add_columns(df_fr)
df_de = add_columns(df_de)

# Select the best columns based on correlation
best_features_fr = select_features_based_on_correlation(df_fr, 'TARGET', 0.8, 0.08)
best_features_de = select_features_based_on_correlation(df_de, 'TARGET', 0.8, 0.15)

# Keep the selected columns while adding the original columns
df_fr = pd.concat([original_fr, df_fr[best_features_fr+['TARGET']]], axis=1)
df_de = pd.concat([original_de, df_de[best_features_de+['TARGET']]], axis=1)

In [12]:
# Display dimensions and final columns of the DataFrames
print("Dimensions of the DataFrame for FR:", df_fr.shape)
print("Dimensions of the DataFrame for DE:", df_de.shape)
print("Columns of the DataFrame for FR:", df_fr.columns)
print("Columns of the DataFrame for DE:", df_de.columns)

Dimensions of the DataFrame for FR: (851, 13)
Dimensions of the DataFrame for DE: (643, 21)
Columns of the DataFrame for FR: Index(['FR_WINDPOW', 'GAS_RET', 'CARBON_RET', 'DE_HYDRO', 'DE_WINDPOW',
       'FR_WINDPOW', 'GAS_RET', 'CARBON_RET', 'DE_CONSUMPTION_MIN_30D',
       'FR_HYDRO_MIN_7D', 'DE_SOLAR_EFFECT', 'FR_WIND_SOLAR', 'TARGET'],
      dtype='object')
Columns of the DataFrame for DE: Index(['DE_FR_EXCHANGE', 'DE_NET_IMPORT', 'DE_GAS', 'DE_COAL', 'DE_HYDRO',
       'DE_WINDPOW', 'DE_LIGNITE', 'DE_RESIDUAL_LOAD', 'DE_WIND',
       'DE_NET_IMPORT', 'DE_GAS', 'DE_HYDRO', 'DE_WINDPOW', 'FR_WINDPOW',
       'DE_RESIDUAL_LOAD', 'DE_GAS_SLOPE_7D', 'DE_WINDPOW_MEAN_7D',
       'DE_WINDPOW_SLOPE_7D', 'DE_WIND_SOLAR', 'FR_WIND_SOLAR', 'TARGET'],
      dtype='object')


## Model and training

#### Linear regression basic

In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

scaler = StandardScaler()
model = LinearRegression()

X = df_fr.drop(columns=['TARGET'])
y = df_fr['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
# We compute the Spearman score
print(f"Spearman score: {spearmanr(y_test,y_pred).correlation}")

Spearman score: 0.22683192144382217


In [14]:
X = df_de.drop(columns=['TARGET'])
y = df_de['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
# We compute the Spearman score
print(f"Spearman score: {spearmanr(y_test,y_pred).correlation}")

Spearman score: 0.4149373881932022
